# Chapter 12 &mdash; CFG to PDA: the Goal/Subgoal Stack Machine

**Concept 9 of the Chapter 12 decomposition:** *CFG to PDA Conversion: the Goal/Subgoal Stack Machine*

A three-state PDA: push $S$, repeatedly replace a goal by a right-hand side, match terminals.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-CFG-To-PDA/Concept-CFG-To-PDA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.AnimatePDA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimatePDA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimatePDA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


Every CFG becomes a PDA mechanically, and the resulting machine has just **three
states**:

* $q_0 \xrightarrow{\varepsilon,\,z_0\,;\,S z_0} q_1$ &mdash; push the start symbol;
* for each production $A \to \alpha$: $q_1 \xrightarrow{\varepsilon,\,A\,;\,\alpha} q_1$
  &mdash; **replace a goal by its right-hand side**;
* for each terminal $a$: $q_1 \xrightarrow{a,\,a\,;\,\varepsilon} q_1$ &mdash; **match**
  the input against the top of the stack;
* $q_1 \xrightarrow{\varepsilon,\,z_0\,;\,z_0} q_2$ &mdash; accept when the stack is clean.

The stack holds **goals still to be achieved**, so this is a top-down parser. Its
nondeterminism is exactly the choice of which production to use &mdash; which is why an
ambiguous grammar gives several accepting computations (Concept 10).

## 2. Definitions

### The conversion

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

In [ ]:
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps


def cfg2pda(G):
    lines = ['PDA', "I : '' , # ; %s#  -> P" % G['S']]
    for A in sorted(G['P']):
        for r in G['P'][A]:
            push = ''.join(r) if r else "''"
            lines.append("P : '' , %s ; %-6s -> P    !! %s -> %s"
                         % (A, push, A, push))
    for a in sorted(G['Sigma']):
        lines.append("P : %s , %s ; ''  -> P    !! match %s" % (a, a, a))
    lines.append("P : '' , # ; #  -> F")
    return md2mc('\n'.join(lines))

# --- a thin wrapper over Jove's PDA runner -----------------------------
# run_pda returns (surviving-IDs, accepting-paths, visited-IDs); a string
# is accepted exactly when the list of accepting paths is non-empty.
def pda_accepts(P, s, acceptance='ACCEPT_F', STKMAX=6):
    surv, paths, visited = run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)
    return len(paths) > 0

def pda_npaths(P, s, acceptance='ACCEPT_F', STKMAX=6):
    return len(run_pda(s, P, acceptance=acceptance, STKMAX=STKMAX)[1])

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;8.&nbsp;Designing Practical PDA in Markdown: $L_{abORac}$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Designing-PDA-In-Markdown/Concept-Designing-PDA-In-Markdown.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;10.&nbsp;Disambiguation Measured: 1 Parse versus 36](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Disambiguation-Measured/Concept-Disambiguation-Measured.ipynb)&nbsp;&rarr;

---

## 3. Tests

The grammar, and the three-state machine it becomes.

In [ ]:
G = mkg({'S': ["", "aSb"]})
show(G)
P = cfg2pda(G)
print()
print("PDA states :", sorted(P["Q"]))
assert len(P["Q"]) == 3

The stack holds **goals**. Watch `S` be replaced and terminals matched off.

In [ ]:
surv, paths, visited = run_pda('aabb', P, STKMAX=10)
print("accepted? ", bool(paths))
for idd in paths[0][1]:
    print("   ", idd)
assert paths

The machine's language is the grammar's language.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(5) for p in product('ab', repeat=k)]
L = set(language(G, 4))
bad = [s for s in strs if pda_accepts(P, s, STKMAX=6) != (s in L)]
print("mismatches over %d strings :" % len(strs), bad)
assert not bad

It works for any grammar &mdash; here, Dyck.

In [ ]:
D = mkg({'S': ["", "(S)", "SS"]})
PD = cfg2pda(D)
LD = set(language(D, 4))
strs = [''.join(p) for k in range(5) for p in product('()', repeat=k)]
bad = [s for s in strs if pda_accepts(PD, s, STKMAX=6) != (s in LD)]
print("Dyck : %d states, mismatches %s" % (len(PD["Q"]), bad))
assert not bad

Three states, always &mdash; the grammar lives in $\Gamma$ and $\Delta$, not in $Q$.

In [ ]:
for name, GG in [('a^n b^n', G), ('Dyck', D),
                 ('expressions', mkg({'E': ["T", "E+T"], 'T': ["F", "T*F"],
                                      'F': ["1", "2", "(E)"]}, 'E'))]:
    PP = cfg2pda(GG)
    print("%-14s |Q| = %d, |Gamma| = %2d, |Delta| = %2d"
          % (name, len(PP["Q"]), len(PP["Gamma"]), len(PP["Delta"])))
    assert len(PP["Q"]) == 3

So **every CFL has a PDA** &mdash; one half of the CFG&harr;PDA equivalence.

In [ ]:
print("CFG -> PDA : this construction")
print("PDA -> CFG : a more elaborate construction (see the text)")
print("Together: context-free languages == PDA languages.")

## 4. Animation

The three-state goal machine for $a^nb^n$.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(cfg2pda(mkg({'S': ['', 'aSb']})), FuseEdges=True)

## 5. Exercises


1. Why is the "match" transition needed at all? What would go wrong without it?
2. How many $\Delta$ entries does a grammar with $p$ productions produce?
3. Is the resulting PDA ever deterministic?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-CFG-To-PDA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')